In [ ]:
import os
import numpy as np
import pandas as pd
import pickle as pkl
from tqdm import tqdm
import torch

import hla_genes

In [ ]:
hla_alleles = sorted(hla_genes.HLA_ALLELES, key=lambda x: x.split("_")[0] + x.split("_")[1].zfill(4))
hla_alleles = [ a for a in hla_alleles if "9901" not in a]

HLA_ALLELE_4D_MAPPING = { allele: i for i, allele in enumerate(hla_alleles) }
HLA_ALLELE_4D_MAPPING = { hla_genes.ukbb_helpers.standardize_ukbb_allele_name(k): v for k, v in HLA_ALLELE_4D_MAPPING.items() }
HLA_ALLELE_4D_MAPPING_INV = { v: k for k, v in HLA_ALLELE_4D_MAPPING.items() }

HLA_ALLELE_2D_MAPPING = { allele_2d:i for i, allele_2d in enumerate(sorted(list(set([ k.split(":")[0] for k, v in HLA_ALLELE_4D_MAPPING.items() ])))) }

In [ ]:
# hla_allele_dosages = pd.concat([
#     pd.DataFrame([ allele.split("_") for allele in hla_df.columns.tolist() ], columns=["gene", "allele"]),
#     pd.DataFrame(hla_df.values.T),  
# ], axis=1)
# 
# standardize_allele_name = lambda x: x.gene + "*" + x.allele.zfill(4)[:2] + ":" + "" + x.allele.zfill(4)[2:]
# hla_allele_dosages['std_allele_name'] = hla_allele_dosages.apply(standardize_allele_name, axis=1)

In [ ]:
# all_allele_sequences = hla_genes.get_hla_protein_sequences()
# hla_dosages_df # ['DQB1'].sum(axis=1)

In [ ]:
def inverse_one_hot(df, gene_name):
    result = []
    for _, row in df.iterrows():
        alleles = []
        for allele, count in row.items():
            alleles.extend([allele] * count)
        result.append(alleles)
    return pd.DataFrame(result, columns=[f"{gene_name}_1", f"{gene_name}_2"], index=df.index)
    

def load_hla_genes():

    OUTPUT_FILE = "data/ukb_real_data/allele_tokens_per_genes.pkl"

    if not os.path.exists(OUTPUT_FILE):

        hla_dosages_df = hla_genes.get_dosages_per_gene(create_hierarchical_index=True, debug_mode=False)
    
        allele_tokens = []
        
        for gene in hla_dosages_df.columns.get_level_values(0).unique():
            
            # print(gene)
            dosage_for_gene = hla_dosages_df[gene]
        
            if gene in { "DRB3", "DRB4", "DRB5" }:
                dosage_for_gene = dosage_for_gene.drop('99', axis=1)
        
            allele_tokens.append(inverse_one_hot(dosage_for_gene, gene))
        
        allele_tokens = pd.concat(allele_tokens, axis=1)
        pkl.dump(allele_tokens, open(OUTPUT_FILE, "wb"))
        
    else:
        allele_tokens = pkl.load(open(OUTPUT_FILE, "rb"))
        return allele_tokens

In [ ]:
merged_df.drop("ethnicity",axis=1).groupby(["broad_ethnicity", "gene", "serotype", "subtype"]).sum()

In [ ]:

merged_df = ethnicity_df.merge(hla_dosages_df, left_index=True, right_index=True)

In [ ]:
hla_dosages_df.sum(axis=0).reset_index().set_axis(["locus", "allele_group", "subtype", "counts"], axis=1).to_csv("allele_counts_2_fields.csv", index=False)

In [ ]:
(allele_tokens_4_digits := load_hla_genes()).head()

In [ ]:
(allele_tokens_2_digits := allele_tokens_4_digits.applymap(lambda x: x if x is None else x[0])).head()

In [ ]:
ORIG_LABELS_FILE = f"data/ukb_simulated_data/labels.csv"
labels = pd.read_csv(ORIG_LABELS_FILE, header=None, sep="\t", names=["label"])
i2l_pre = { index: label[0] for index, label in labels.iterrows()}

def update_vocab(i2l_mapping, hla_allele_mapping):
    pre = {k: v for k, v in i2l_mapping.items() if k <= 3}
    post = {k: v for k, v in i2l_mapping.items() if k > 3}
    hla_tokens = {i + 4: hla for hla, i in hla_allele_mapping.items()}
    post_shifted = {k + len(hla_allele_mapping): v for k, v in post.items()}
    new_vocab = {**pre, **hla_tokens, **post_shifted}

    return new_vocab

In [ ]:
## Re-generate labels files

In [ ]:
len(HLA_ALLELE_4D_MAPPING)

In [ ]:
HLA_ALLELE_MAPPING = HLA_ALLELE_2D_MAPPING

In [ ]:
def modify_labels_with_hla(hla_allele2index_mapping):
    
    i2l = update_vocab(i2l_pre, hla_allele2index_mapping)
    l2i = { l: i for i, l in i2l.items() }

    n_alleles = len(hla_allele2index_mapping)

    pre2post_mapping = { k-1: l2i[v]-1 for k, v in i2l_pre.items() }

    df = pd.read_csv("delphi_labels_chapters_colours_icd.csv")
    df_birth = df.iloc[:4,:]
    df_rest = df.iloc[4:,:]

    hla_data = []
    for index, hla_allele in { k:v for k,v in i2l.items() if "HLA" in v }.items():
        hla_data.append([index, hla_allele, None, "HLA", "HLA", "#000000"])
    df_hla = pd.DataFrame(hla_data, columns=df_birth.columns)

    df_rest['index'] = df_rest['index'] + n_alleles
    return pd.concat([df_birth, df_hla, df_rest]).set_index("index"), pre2post_mapping, i2l, l2i


# def modify_labels_with_hla_4digits():
# 
#     i2l = update_vocab(i2l_pre, HLA_ALLELE_4D_MAPPING)
#     l2i = { l: i for i, l in i2l.items() }
# 
#     pre2post_mapping = { k-1: l2i[v]-1 for k, v in i2l_pre.items() }
# 
#     df = pd.read_csv("delphi_labels_chapters_colours_icd.csv")
#     df_birth = df.iloc[:4,:]
#     df_rest = df.iloc[4:,:]
# 
#     hla_data = []
#     for index, hla_allele in { k:v for k,v in i2l.items() if "HLA" in v }.items():
#         hla_data.append([index, hla_allele, None, "HLA", "HLA", "#000000"])
#     df_hla = pd.DataFrame(hla_data, columns=df_birth.columns)
# 
#     df_rest['index'] = df_rest['index'] + 359
#     return pd.concat([df_birth, df_hla, df_rest]).set_index("index"), pre2post_mapping, i2l, l2i

In [ ]:
labels_2digits, pre2post_mapping_2digits, i2l_2digits, l2i_2digits = modify_labels_with_hla(HLA_ALLELE_2D_MAPPING)
labels_2digits.to_csv("delphi_labels_chapters_colours_icd_with_hla2d.csv")
labels_2digits.name.to_frame()# .to_csv("data/ukb_real_data_2digit/labels.csv", index=False, header=False)

labels_4digits, pre2post_mapping_4digits, i2l_4digits, l2i_4digits = modify_labels_with_hla(HLA_ALLELE_4D_MAPPING)
labels_4digits.to_csv("delphi_labels_chapters_colours_icd_with_hla4d.csv")
labels_4digits.name.to_frame().to_csv("data/ukb_real_data_4digit/labels.csv", index=False, header=False)

In [ ]:
l2i = l2i_2digits
pre2post_mapping = pre2post_mapping_2digits

l2i = l2i_4digits 
pre2post_mapping = pre2post_mapping_4digits

In [ ]:
alleles_for_pop = {}

for subject_id, row in tqdm(allele_tokens.iterrows()):
    alleles_for_pop[subject_id] = (alleles_for_subject := [])
    for i, x in enumerate(row):
        if x is None:
            continue        
        # allele = "HLA-" + row.index[i].split("_")[0] + "*" + x # for 2 digits
        allele = "HLA-" + row.index[i].split("_")[0] + "*" + ":".join(x) # for 4 digits
        # print(f"{allele=}")
        alleles_for_subject.append(l2i[allele])

In [ ]:
dataset, file_prefix = 'ukb_real_data', "ukb_real_"
# dataset, file_prefix = 'ukb_simulated_data', ''

data_dir = os.path.join('data', dataset)
train_data = np.memmap(os.path.join(data_dir, f'{file_prefix}train.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)
val_data = np.memmap(os.path.join(data_dir, f'{file_prefix}val.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)

train_ids = set(np.unique(train_data[:,0]))
val_ids   = set(np.unique(val_data[:,0]))

In [ ]:
pre2post_mapping = pre2post_mapping_2digits
pre2post_mapping = pre2post_mapping_4digits

# train_data = np.array([ [row[0], row[1], pre2post_mapping[row[2]]] for row in np.array(train_data) ])
# val_data = np.array([ [row[0], row[1], pre2post_mapping[row[2]]] for row in np.array(val_data) ])

In [ ]:
train_allele_bin = []
val_allele_bin = []

for subject_id, allele_indices in tqdm(alleles_for_pop.items()):
    if int(subject_id) in train_ids:
        train_allele_bin.extend([[int(subject_id), 0, i] for i in allele_indices ])
    if int(subject_id) in val_ids:
        val_allele_bin.extend([[int(subject_id), 0, i] for i in allele_indices ])

train_allele_bin = np.array(train_allele_bin)
val_allele_bin = np.array(val_allele_bin)

train_allele_bin = np.concatenate([train_data, train_allele_bin])
val_allele_bin = np.concatenate([val_data, val_allele_bin])

train_data = pd.DataFrame(train_allele_bin, columns=['subject_id', 'age', 'event']).sort_values(['subject_id', 'age']).values
val_data   = pd.DataFrame(val_allele_bin, columns=['subject_id', 'age', 'event']).sort_values(['subject_id', 'age']).values
train_data = np.array(train_data, dtype=np.uint32)
val_data = np.array(val_data, dtype=np.uint32)

In [ ]:
# train_data.tofile("data/ukb_real_data_2digit_5fold/train.bin")
# val_data.tofile("data/ukb_real_data_2digit_5fold/val.bin")

train_data.tofile("data/ukb_real_data_4digit/train.bin")
val_data.tofile("data/ukb_real_data_4digit/val.bin")

In [ ]:
train_ids = sorted(list(set(train_data[:,0])))
train_data_len = len(train_ids) 

folds_boundaries = zip(np.arange(0, 1, 0.25) * train_data_len, np.arange(0.25, 1.25, 0.25) * train_data_len)

fold_data = []

for imin, imax in folds_boundaries:
    print(imin, imax)
    imin = int(imin)
    imax = int(imax) - 1
    fold_idmin, fold_idmax = train_ids[imin], train_ids[imax]
    print(fold_idmin, fold_idmax)
    fold_data.append(train_data[(train_data[:,0] >= fold_idmin) & (train_data[:,0] <= fold_idmax)])

fold_data.append(val_data)

In [ ]:
for i in range(5):
    fold_data[i].tofile(f"data/ukb_real_5_folds_nohla/fold{i+1}.bin")

In [ ]:
(train_data[:,0] >= 1000015) & (train_data[:,0] < 1001000)

In [ ]:
# [ subject_id for subject_id, age, id in row for row in np.array(train_data) ] 
# train_data[train_data[:, 0] == 1000015] #[:20]
# [ len(x) for x in alleles_for_pop if len(x) > 18 ]

In [ ]:
class HLAAlleleOneHotEncoding(torch.nn.Module):

    def __init__(self, num_embeddings, embedding_dim):
        
        super().__init__()
        self.hla_embedding = torch.nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)

    def forward(self, x):
        return self.hla_embedding(x)        


class ESMEmbedding(torch.nn.Module):

    def __init__(self, embedding_file, alleles=None, metadata=None, apply_pca=False):

        super().__init__()
        self.embedding_table = pkl.load(open(embedding_file, "rb"))
        self.metadata = metadata
        # self.embedding = torch.nn.Embedding()
        if apply_pca:
            raise NotImplementedError
        
    def forward(self, x):
        
        return self.embedding(x)

In [ ]:
onehot_hla_embedding = HLAAlleleOneHotEncoding(num_embeddings=362, embedding_dim=120)
esm_emb = ESMEmbedding("../hla_genes/cache/hla_protseq_embeddings_esm2_t12_35M_UR50D.pkl")